# Bivariate Exploratory Data Analysis (EDA) — FT draft v2

**Week 5 · Monday · Part 2 · Estimated classroom time: 85–90 minutes**

**Business question:** How do housing characteristics relate to sale prices, and how can we explore relationships between two variables?

In the **Univariate EDA** notebook, we studied one variable at a time. Here we study **two variables together**. We will focus on **descriptive exploration**, not statistical significance tests.

**Learning goals:**
- Choose an appropriate summary and visualization for each pair of variable types.
- Compare **absolute counts first**, then proportions, keeping the group sizes visible.
- Describe differences between numerical distributions across categories.
- Identify the limitations of small groups and nonrepresentative samples.
- Distinguish patterns in data from claims about causality.

## 1. Setup and the three types of bivariate analysis (5 minutes)

| Variable 1 | Variable 2 | Useful tools |
|---|---|---|
| Categorical | Categorical | Crosstab, grouped/stacked bars, frequency heatmap |
| Categorical | Numerical | Group summaries, boxplot, violin plot, mean bar chart |
| Numerical | Numerical | Scatter plot, optional regression line |

We reuse the **same housing dataset** as in the Univariate EDA notebook. This notebook can also run independently.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline
sns.set_theme(style="whitegrid")

DATA_URL = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/housing_price_eda.csv"
df = pd.read_csv(DATA_URL)

df[["MSZoning", "SaleCondition", "SalePrice", "GrLivArea", "CentralAir"]].head()

## 2. Categorical vs. categorical: zoning and sale condition (25 minutes)

**Business question:** Is the mix of sale conditions similar across zoning classifications?

### 2.1. Start with absolute frequencies (`pd.crosstab`)

A **contingency table** counts how many observations belong to each combination of categories. Each cell is a count, not a probability.

**Always check how many observations are available in each group before comparing percentages.**

In [ ]:
counts = pd.crosstab(df["MSZoning"], df["SaleCondition"])

# Marginal totals reveal how many houses belong to each zoning group.
counts_with_totals = pd.crosstab(
    df["MSZoning"], df["SaleCondition"], margins=True
)
counts_with_totals

### 2.2. Then compare relative frequencies

Absolute counts describe **how many** houses we observed. Row proportions describe **the composition within each zoning group**:

*Among the houses observed in this zoning category, what fraction belongs to each sale condition?*

We normalize by row because this is the question we want to answer. Normalizing by column would answer a **different** question.

**Important:** A percentage from a group of 2 houses is much less stable than the same percentage from a group of 200 houses. Percentages should be read together with the underlying counts.

In [ ]:
row_proportions = pd.crosstab(
    df["MSZoning"],
    df["SaleCondition"],
    normalize="index"
)

(row_proportions * 100).round(1)

**Interpretation:** Raw counts describe **volume**; row percentages describe **composition within each zoning group**. Neither tells us whether an association is statistically significant.

### 2.3. Grouped bar chart

The `hue` argument introduces a second categorical variable into the visualization. This chart preserves the **absolute group sizes**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=df, x="MSZoning", hue="SaleCondition", ax=ax)
ax.set(title="Sale conditions across zoning categories", xlabel="Zoning", ylabel="Number of homes")
plt.show()

### 2.4. Proportional stacked bars

A grouped count plot emphasizes group sizes. A **100% stacked bar chart** makes category compositions easier to compare, even when the groups have different sizes.

**But the bars all have the same height, even if one group has only a handful of observations.** We will label each group with its sample size (`n`) to avoid hiding this limitation.

In [ ]:
# Row proportions were calculated from the absolute-frequency table above.
ax = row_proportions.plot(kind="bar", stacked=True, figsize=(9, 5))

# Show the number of observations behind each percentage bar.
group_sizes = counts.sum(axis=1).reindex(row_proportions.index)
ax.set_xticklabels(
    [f"{category} (n={n})" for category, n in group_sizes.items()],
    rotation=0
)
ax.set(
    title="Composition of sale conditions by zoning",
    xlabel="Zoning (observed group size)",
    ylabel="Proportion"
)
ax.legend(title="Sale condition", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 2.5. Frequency heatmap

Color is another way to display a contingency table. Notice that a heatmap of **counts** and one of **proportions** answer different questions.

Here we deliberately display **absolute counts**, so small groups remain visible.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(counts, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set(title="Number of homes by zoning and sale condition", xlabel="Sale condition", ylabel="Zoning")
plt.tight_layout()
plt.show()

### Check for understanding · 4 minutes

1. Which zoning category contains the most observations? Use the **absolute-frequency table** to justify your answer.
2. Which zoning categories appear different when you compare **row percentages**?
3. Imagine a category has only 2 houses and both share the same sale condition. Would you expect its observed percentage to be stable if we collected more houses? Explain.

**Optional:** Change the heatmap to display percentages using `row_proportions`, `fmt=".1%"` and an appropriate color scale. Keep the counts table available for interpretation.

### A crucial limitation: is this dataset representative?

Even a large sample is not necessarily representative of all houses in a city. A real-estate company might specialize in certain types of properties, and many owners may never contact that company. As a result, some homes could be systematically missing from its records (**selection bias**).

**Small groups require caution.** Percentages calculated from very few observations can change substantially when new data are collected. Always inspect group sizes before interpreting proportions, and remember that a large dataset is not necessarily representative of the population.

More observations generally reduce random sampling variability **when the sampling process is appropriate**; they do not automatically correct selection bias.

## 3. Categorical vs. numerical: zoning and sale price (30 minutes)

**Business question:** Do typical house prices and their variability differ across zoning categories?

### 3.1. Summarize each group

We need both **central tendency** and **dispersion**. A group mean by itself does not describe the full distribution.

In [ ]:
price_by_zone = (
    df.groupby("MSZoning")["SalePrice"]
    .agg(count="count", mean="mean", median="median", std="std")
    .sort_values("median", ascending=False)
)
price_by_zone.round(1)

### 3.2. Side-by-side boxplots

Boxplots make it easy to compare medians, quartiles, spread and **potentially unusual observations**. As discussed in Univariate EDA, Tukey whiskers are a **screening convention**, not a rule for deleting data.

In [ ]:
zone_order = price_by_zone.index
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="MSZoning", y="SalePrice", order=zone_order, ax=ax)
ax.set(title="Sale price distribution by zoning", xlabel="Zoning", ylabel="Sale price ($)")
plt.show()

### 3.3. Violin plots and the KDE idea

A violin plot adds a smoothed view of the distribution within each group. The **width of the violin** reflects estimated density, not the number of observations by default. Be cautious with very small groups: their estimated shapes may change substantially if we collect more observations.

**Before interpreting a violin plot, check the `count` column in the group summary above.**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(data=df, x="MSZoning", y="SalePrice", order=zone_order, inner="quart", ax=ax)
ax.set(title="Sale price distribution by zoning", xlabel="Zoning", ylabel="Sale price ($)")
plt.show()

### 3.4. Mean bar charts: what do they hide?

Seaborn's `barplot` summarizes the **mean** by default. A bar showing a mean does **not** show the spread of individual prices or the size of the group. Here we disable error bars to focus on this limitation; the boxplot is more informative for comparing distributions.

Always interpret the chart together with the group summary (`count`, `mean`, `median`, `std`).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=df, x="MSZoning", y="SalePrice", order=zone_order, errorbar=None, ax=ax)
ax.set(title="Average sale price by zoning", xlabel="Zoning", ylabel="Mean sale price ($)")
plt.show()

### Check for understanding · 4 minutes

1. Is the zoning category with the highest **mean** price also the one with the highest **median**?
2. Which chart would you use to compare **variability** rather than just average price?
3. Which zoning groups have few observations? Why might their mean, median or violin shape change substantially if new houses were added?
4. Could a real-estate company's portfolio be systematically different from all homes in the city? Give one reason.

## 4. Numerical vs. numerical: living area and sale price (20 minutes)

**Business question:** Do larger houses tend to sell for higher prices?

### 4.1. Scatter plot

Each point represents **one house**. We look for direction, shape, strength, clusters and unusual observations. A scatter plot does not, by itself, establish causation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", alpha=0.6, ax=ax)
ax.set(title="Sale price vs. above-ground living area", xlabel="Living area (sq ft)", ylabel="Sale price ($)")
plt.show()

### 4.2. A fitted line as a visual guide

`regplot` can overlay a straight line fitted to the observations. The line is a **descriptive summary**, not proof that the relationship is causal or that a linear model is appropriate. We will study statistical inference for correlation later in the week.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.regplot(
    data=df,
    x="GrLivArea",
    y="SalePrice",
    scatter_kws={"alpha": 0.4},
    line_kws={"color": "darkred"},
    ci=None,
    ax=ax
)
ax.set(title="Living area and sale price: descriptive fitted line", xlabel="Living area (sq ft)", ylabel="Sale price ($)")
plt.show()

### 4.3. Adding a category with `hue`

Coloring points by `CentralAir` may reveal groups worth investigating. However, differences in a scatter plot can reflect other factors (such as location, quality or house age).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", hue="CentralAir", alpha=0.6, ax=ax)
ax.set(title="Living area, price and central air", xlabel="Living area (sq ft)", ylabel="Sale price ($)")
plt.show()

### Check for understanding · 4 minutes

1. Describe the overall direction of the relationship between living area and sale price.
2. Identify one observation that might deserve further investigation. Would you remove it immediately? Why not?
3. Does this plot prove that increasing a house's living area **causes** a specific increase in its price? What other variables might matter?

## 5. Wrap-up: choose the right comparison (5 minutes)

For a business report, use this structure:

1. **Question:** Which two variables are we comparing?
2. **Evidence:** Which table, statistic or plot supports the observation? How many observations are in each group?
3. **Interpretation:** What pattern do we see, and how strong is the descriptive evidence?
4. **Caution:** Are group sizes, skewness, unusual observations, selection bias or hidden variables important?

**Mini-task:** Write a two-sentence insight comparing sale prices across zoning groups, supported by one statistic and one visual observation. Add one caveat about the reliability or representativeness of the evidence.

### What comes later this week?

- **Wednesday:** Chi-square test of independence and Cramér's V.
- **Thursday:** Pearson/Spearman correlation, correlation heatmaps and Q-Q plots.

**Key idea:** A convincing chart is not enough. We must also ask whether the data justify the conclusion.